# Hindcast verification and precursor metrics

Raw inputs are read from `PAPER1_ARCHIVE_ROOT`; preprocessing products are read from `PAPER1_PREPROCESSED_ROOT`; new diagnostics are written to `PAPER1_DERIVED_ROOT` (default: repository-local `work/`). Source files are never modified.


## Reference definitions

Inputs: canonical BWCN O3, dynamics, NAM/AO. Outputs: in-memory year-0008 reference arrays. Method: select model year 0008 explicitly by coordinates; define centered-five-day Mar1--Apr30 scoring.


In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

def discover_diagnostic_directory():
    candidates = (
        Path.cwd(), Path.cwd() / "analysis",
        Path.cwd() / "Paper1" / "analysis",
    )
    for candidate in candidates:
        if (candidate / "lib" / "workflow_io.py").is_file():
            return candidate.resolve()
    raise FileNotFoundError(
        "Cannot locate Paper1/analysis/lib from the current working directory"
    )

NOTEBOOK_DIR = discover_diagnostic_directory()
LIB = NOTEBOOK_DIR / "lib"
if str(LIB) not in sys.path:
    sys.path.insert(0, str(LIB))

from workflow_io import (
    PRODUCT_VERSION, archive_root, derived_root, preprocessed_root, product_path,
    write_csv_atomic, write_netcdf_atomic,
)

ARCHIVE_ROOT = archive_root()
PREPROCESSED_ROOT = preprocessed_root()
DERIVED_ROOT = derived_root()
MARCH_HINDCAST_ROOT = Path(os.environ.get(
    "PAPER1_MARCH_HINDCAST_SOURCE",
    ""
    "",
))
OVERWRITE = os.environ.get("PAPER1_OVERWRITE_STAGING", "0") == "1"
print("read-only archive root:", ARCHIVE_ROOT)
print("preprocessed staging input root:", PREPROCESSED_ROOT)
print("read-only March hindcast root:", MARCH_HINDCAST_ROOT)
print("staging output root:", DERIVED_ROOT)
print("diagnostic notebook directory:", NOTEBOOK_DIR)

CASE_DISPLAY_START = {"0008-01": 80101, "0008-02": 80201, "0008-03": 80301}
CASE_DISPLAY_NT = {"0008-01": 151, "0008-02": 120, "0008-03": 92}
DISPLAY_END_DATE = 80531
NOLEAP_MONTH_LENGTHS = (31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31)

def noleap_date_range(start_date, end_date):
    start_date, end_date = int(start_date), int(end_date)
    start_year, end_year = start_date // 10000, end_date // 10000
    if start_year != end_year:
        raise ValueError("Display-window helper requires one no-leap model year")
    dates = []
    for month, length in enumerate(NOLEAP_MONTH_LENGTHS, start=1):
        for day in range(1, length + 1):
            value = start_year * 10000 + month * 100 + day
            if start_date <= value <= end_date:
                dates.append(value)
    return np.asarray(dates, dtype=int)

def require_case_display_window(case, dates):
    dates = np.asarray(dates, dtype=int)
    if dates.ndim != 1 or np.unique(dates).size != dates.size or np.any(np.diff(dates) <= 0):
        raise RuntimeError(f"{case}: dates must be one-dimensional, unique, and strictly increasing")
    expected = noleap_date_range(CASE_DISPLAY_START[case], DISPLAY_END_DATE)
    if expected.size != CASE_DISPLAY_NT[case]:
        raise AssertionError(f"{case}: internal display-day count is wrong")
    lookup = {int(value): index for index, value in enumerate(dates)}
    missing = [int(value) for value in expected if int(value) not in lookup]
    if missing:
        raise RuntimeError(
            f"{case}: display window is incomplete through 0008-05-31; "
            f"missing {missing[:5]}"
        )
    indices = np.asarray([lookup[int(value)] for value in expected], dtype=int)
    return expected, indices

def require_finite_display_values(context, values, indices, *, time_axis):
    selected = np.take(np.asarray(values, dtype=float), indices, axis=time_axis)
    finite = np.isfinite(selected)
    if not bool(finite.all()):
        raise RuntimeError(
            f"{context}: display window through 0008-05-31 has "
            f"{int(finite.size - finite.sum())} non-finite value(s)"
        )

from paper1_diagnostics import crps_ensemble, sign_agreement

CASES = ("0008-01", "0008-02", "0008-03")

def align_reference(field, field_dates, requested_dates):
    lookup = {int(date): index for index, date in enumerate(np.asarray(field_dates, dtype=int))}
    try:
        indices = [lookup[int(date)] for date in requested_dates]
    except KeyError as error:
        raise RuntimeError(f"Reference year 0008 is missing restart date {error.args[0]}") from error
    return np.asarray(field.values, dtype=float)[indices]

with xr.open_dataset(product_path("ozone", "bwcn_partial_o3.nc"), decode_times=False) as source:
    mask = np.asarray(source.model_year.values, dtype=int) == 8
    reference_o3 = source.partial_o3_du.isel(time=np.flatnonzero(mask)).load()
    reference_o3_dates = np.asarray(source.date.values, dtype=int)[mask]
with xr.open_dataset(product_path("dynamics", "waccm_bwcn_year0008.nc"), decode_times=False) as source:
    reference_dynamics = source.load()
with xr.open_dataset(product_path("nam", "waccm_bwcn_year0008_nam_ao.nc"), decode_times=False) as source:
    reference_nam = source.load()


## Load aligned 30-member cases

Inputs: staged O3, dynamics, and NAM/AO for January, February, and March. Outputs: one coordinate-checked in-memory case dictionary. Method: require exactly 30 members, identical unique integer dates, every daily value finite, and uninterrupted display coverage from each initialization through 0008-05-31 (151/120/92 days).


In [ ]:
from relationship_products import centered_spring_detail

master = pd.read_csv(product_path("ozone", "waccm_master_rankings.csv"))
master_threshold = float(master.low25_threshold_du.iloc[0])
verification_cases = {}
for case in CASES:
    with xr.open_dataset(product_path("ozone", f"hindcast_{case}_partial_o3.nc"), decode_times=False) as source:
        dates = np.asarray(source.date.values, dtype=int)
        members = np.asarray(source.member.values).astype(str)
        o3 = np.asarray(source.partial_o3_du.values, dtype=float)
    with xr.open_dataset(product_path("dynamics", f"hindcast_{case}.nc"), decode_times=False) as source:
        if not np.array_equal(dates, np.asarray(source.date.values, dtype=int)):
            raise RuntimeError(f"{case}: O3 and dynamics dates differ")
        if not np.array_equal(members, np.asarray(source.member.values).astype(str)):
            raise RuntimeError(f"{case}: O3 and dynamics member order differs")
        dynamics = source.load()
    with xr.open_dataset(product_path("nam", f"hindcast_{case}_nam_ao.nc"), decode_times=False) as source:
        if not np.array_equal(dates, np.asarray(source.date.values, dtype=int)):
            raise RuntimeError(f"{case}: O3 and NAM dates differ")
        if not np.array_equal(members, np.asarray(source.member.values).astype(str)):
            raise RuntimeError(f"{case}: O3 and NAM member order differs")
        nam_source = source.load()
    fields = {
        "o3": o3, "u60n10": np.asarray(dynamics.u60n10.values, dtype=float),
        "tmin50": np.asarray(dynamics.tmin50.values, dtype=float),
        "ep100": np.asarray(dynamics.ep100_upward_40_80n.values, dtype=float),
        "ao": np.asarray(nam_source.ao.values, dtype=float),
        "nam": np.asarray(nam_source.nam.values, dtype=float),
    }
    references = {
        "o3": align_reference(reference_o3, reference_o3_dates, dates),
        "u60n10": align_reference(reference_dynamics.u60n10, reference_dynamics.date, dates),
        "tmin50": align_reference(reference_dynamics.tmin50, reference_dynamics.date, dates),
        "ep100": align_reference(reference_dynamics.ep100_upward_40_80n, reference_dynamics.date, dates),
        "ao": align_reference(reference_nam.ao, reference_nam.date, dates),
        "nam": align_reference(reference_nam.nam, reference_nam.date, dates),
    }
    if any(values.shape[0] != 30 for values in fields.values()):
        raise RuntimeError(f"{case}: every verification field must have exactly 30 members")
    display_dates, display_indices = require_case_display_window(case, dates)
    for name, values in fields.items():
        require_finite_display_values(
            f"{case} member {name}", values, display_indices, time_axis=1,
        )
    for name, values in references.items():
        require_finite_display_values(
            f"{case} reference {name}", values, display_indices, time_axis=0,
        )
    verification_cases[case] = {
        "dates": dates, "members": members, "plev": np.asarray(nam_source.plev.values),
        "display_dates": display_dates, "display_indices": display_indices,
        "fields": fields, "references": references, "metrics": {},
    }


## Ensemble mean

Inputs: six member fields in the aligned case dictionary. Outputs: daily ensemble means. Method: arithmetic member mean with NaN-aware reduction.


In [ ]:
for case_data in verification_cases.values():
    for name, values in case_data["fields"].items():
        case_data["metrics"][f"{name}_ensemble_mean"] = np.nanmean(values, axis=0)


## Population spread

Inputs: the same six member fields. Outputs: daily spread, including EP100. Method: population standard deviation with ddof=0.


In [ ]:
for case_data in verification_cases.values():
    for name in ("o3", "u60n10", "tmin50", "ep100", "ao"):
        values = case_data["fields"][name]
        case_data["metrics"][f"{name}_spread"] = np.nanstd(values, axis=0, ddof=0)


## Daily CRPS

Inputs: each 30-member field and the coordinate-aligned BWCN year-0008 reference. Outputs: daily CRPS. Method: the Methods-V7 Hersbach ensemble formula.


In [ ]:
for case_data in verification_cases.values():
    for name in ("o3", "u60n10", "tmin50"):
        values = case_data["fields"][name]
        case_data["metrics"][f"{name}_crps"] = crps_ensemble(values, case_data["references"][name])


## Ninety-percent sign agreement

Inputs: the 30 member NAM values for each case, day, and pressure. Outputs: sign-agreement fraction and robust mask. Method: require at least 0.90 of members to share that experiment ensemble-mean NAM sign; no reference-difference branch is used, and a zero ensemble-mean sign is undefined/non-robust.


In [ ]:
for case_data in verification_cases.values():
    values = case_data["fields"]["nam"]
    agreement = sign_agreement(values)
    case_data["metrics"]["nam_sign_agreement"] = agreement
    case_data["metrics"]["nam_robust"] = (np.isfinite(agreement) & (agreement >= 0.90)).astype(np.int8)


## RMSE and epsilon-min

Inputs: January member/reference fields and exact ozone spring series. Outputs: verification/member_metrics.csv. Method: evaluate Figure-7 O3/U60N10/Tmin50 RMSE on exactly 0008-01-01--0008-05-30 (150 complete daily values); epsilon_min remains the member-minus-reference centered-five-day Mar1--Apr30 minimum.


In [ ]:
member_rows = []
reference_detail = centered_spring_detail(
    np.asarray(reference_o3.values), reference_o3_dates,
    np.asarray(reference_o3.values), reference_o3_dates,
)
for case, case_data in verification_cases.items():
    if case != "0008-01":
        continue
    evaluation_mask = (
        (np.asarray(case_data["dates"], dtype=int) >= 80101)
        & (np.asarray(case_data["dates"], dtype=int) <= 80530)
    )
    if int(evaluation_mask.sum()) != 150:
        raise RuntimeError(
            "January Figure-7 RMSE window must contain every no-leap day from 0008-01-01 through 0008-05-30 (Nt=150)"
        )
    for member_index, member in enumerate(case_data["members"]):
        row = {
            "case": case, "member": member, "evaluation_start": "00080101",
            "evaluation_end": "00080530", "evaluation_nt": 150,
        }
        for name, output_name in (
            ("o3", "o3_rmse_du"), ("u60n10", "u60n10_rmse_ms"),
            ("tmin50", "tmin50_rmse_k"),
        ):
            difference = (
                case_data["fields"][name][member_index, evaluation_mask]
                - case_data["references"][name][evaluation_mask]
            )
            finite_nt = int(np.isfinite(difference).sum())
            if finite_nt != 150:
                raise RuntimeError(
                    f"{case} member {member} {name}: RMSE window has {finite_nt}/150 finite daily differences"
                )
            row[f"{name}_finite_nt"] = finite_nt
            row[output_name] = float(np.sqrt(np.mean(difference * difference)))
        detail = centered_spring_detail(
            case_data["fields"]["o3"][member_index], case_data["dates"],
            np.asarray(reference_o3.values), reference_o3_dates,
        )
        row.update(
            minimum_du=detail["minimum_du"], minimum_date=detail["minimum_date"],
            minimum_doy=detail["minimum_doy"],
            epsilon_min_du=detail["minimum_du"] - reference_detail["minimum_du"],
            is_low_o3_by_waccm_master_threshold=detail["minimum_du"] <= master_threshold,
        )
        member_rows.append(row)
member_metrics = pd.DataFrame(member_rows)
write_csv_atomic(
    member_metrics, product_path("verification", "member_metrics.csv"),
    required_columns=[
        "case", "member", "o3_rmse_du", "u60n10_rmse_ms", "tmin50_rmse_k",
        "evaluation_start", "evaluation_end", "evaluation_nt",
        "o3_finite_nt", "u60n10_finite_nt", "tmin50_finite_nt",
        "minimum_du", "minimum_date",
        "minimum_doy", "epsilon_min_du", "is_low_o3_by_waccm_master_threshold",
    ], exact_rows=30, overwrite=OVERWRITE,
)


## Atomic daily verification products

Inputs: separately computed mean, spread, CRPS, and sign metrics. Outputs: verification/{case}_daily.nc including direct reference arrays and explicit display-window metadata. Method: require finite non-sign metrics through 0008-05-31, schema-validate a temporary file, then atomically replace the staging product.


In [ ]:
for case, case_data in verification_cases.items():
    variables = {}
    required = {}
    metric_contract = {
        "o3": ("ensemble_mean", "spread", "crps"),
        "u60n10": ("ensemble_mean", "spread", "crps"),
        "tmin50": ("ensemble_mean", "spread", "crps"),
        "ep100": ("ensemble_mean", "spread"),
        "ao": ("ensemble_mean", "spread"),
        "nam": ("ensemble_mean", "sign_agreement", "robust"),
    }
    for name, metrics in metric_contract.items():
        dims = ("time", "plev") if name == "nam" else ("time",)
        variables[f"{name}_reference"] = (dims, case_data["references"][name])
        required[f"{name}_reference"] = dims
        for metric in metrics:
            variable = f"{name}_{metric}"
            variables[variable] = (dims, case_data["metrics"][variable])
            required[variable] = dims
    daily = xr.Dataset(
        variables,
        coords={"time": np.arange(len(case_data["dates"])), "date": ("time", case_data["dates"]), "plev": case_data["plev"]},
        attrs={
            "product_version": PRODUCT_VERSION, "case": case, "reference": "BWCN year 0008",
            "spread_ddof": 0, "sign_agreement_threshold": 0.9,
            "agreement_basis": "ensemble NAM sign",
            "crps_method": "Hersbach ensemble formula in Methods V7",
            "epflux_method": "do_ubar=True; monthly natural-calendar N2; w=None; wave=-1; upward=-ep2; 40--80N cosine",
            "march_centered5_boundary": "reference BWCN year0008 used only outside restart coverage",
            "display_start_date": f"{int(case_data['display_dates'][0]):08d}",
            "display_end_date": f"{int(case_data['display_dates'][-1]):08d}",
            "display_day_count": int(case_data["display_dates"].size),
            "display_window_complete": "True",
        },
    )
    daily.plev.attrs.update(units="hPa", positive="down")
    for variable in daily.data_vars:
        if variable.endswith(("_sign_agreement", "_robust")):
            continue
        require_finite_display_values(
            f"{case} verification {variable}", daily[variable].values,
            case_data["display_indices"], time_axis=0,
        )
    write_netcdf_atomic(
        daily, product_path("verification", f"{case}_daily.nc"),
        required_vars=required, required_coords=("date", "plev"),
        required_attrs={
            "reference": "BWCN year 0008", "sign_agreement_threshold": 0.9,
            "agreement_basis": "ensemble NAM sign",
            "display_end_date": "00080531", "display_window_complete": "True",
            "display_day_count": int(case_data["display_dates"].size),
        },
        overwrite=OVERWRITE,
    )


## Figure-7 Pearson/OLS links

Inputs: January member RMSE table. Outputs: verification/pearson_links.csv. Method: exactly three accepted Pearson/OLS relations with r, p, n, slope, and intercept.


In [ ]:
from scipy.stats import linregress

pearson_rows = []
january = member_metrics.loc[member_metrics.case == "0008-01"]
for x_metric, y_metric in (
    ("u60n10_rmse_ms", "o3_rmse_du"),
    ("tmin50_rmse_k", "u60n10_rmse_ms"),
    ("tmin50_rmse_k", "o3_rmse_du"),
):
    valid = january[[x_metric, y_metric]].dropna()
    fit = linregress(valid[x_metric], valid[y_metric])
    pearson_rows.append({
        "case": "0008-01", "x_metric": x_metric, "y_metric": y_metric,
        "r": float(fit.rvalue), "p": float(fit.pvalue), "n": int(len(valid)),
        "slope": float(fit.slope), "intercept": float(fit.intercept),
    })
write_csv_atomic(
    pd.DataFrame(pearson_rows), product_path("verification", "pearson_links.csv"),
    required_columns=["case", "x_metric", "y_metric", "r", "p", "n", "slope", "intercept"],
    exact_rows=3, overwrite=OVERWRITE,
)


## Figures 8, 9, and 11 precursor products

Inputs: canonical hindcast O3/EP100 plus the 230-event threshold. Outputs: figure-ready histogram, scatter, RMSE-link, and window-scan products. Method: precompute every minimum, fixed-low25 flag, exact 20-day window mean, Pearson p value, and OLS fit; Figure 8h joins the precomputed 150-day O3 RMSE.


In [ ]:
from relationship_products import build_hindcast_relationships

build_hindcast_relationships(DERIVED_ROOT, overwrite=OVERWRITE)


## Figure-2 precursor products

Inputs: canonical MERRA-2 rankings and the 207 LONGRUN rows of the combined 230-event WACCM classification master, plus EP100, NAM50, and AO. Outputs: figure02c/02d/02g canonical products. Method: use only LONGRUN as the WACCM relationship population while inheriting its low25 flags and threshold unchanged from the combined master. First map every daily record to a ranked Oct--Sep event; a predecessor year contributes only Oct--Dec to its ranked successor and never becomes an extra sample. Then apply calendar-day EP standardization with ddof=0, declared windows, fixed memberships, and Pearson/OLS.


In [ ]:
from relationship_products import build_free_run_relationships

build_free_run_relationships(DERIVED_ROOT, overwrite=OVERWRITE)


## Figure-4 event context

Inputs: target NAM/AO, partial O3, rankings, and event-profile bootstrap. Outputs: cases/figure04_context.nc. Method: target-excluded month-day climatology and precomputed centered-five-day O3 context on Nov1--May31.


In [ ]:
from relationship_products import build_figure04_context

build_figure04_context(DERIVED_ROOT, overwrite=OVERWRITE)
